In [3]:
import sdmx
import pandas as pd
pd.set_option("display.max_rows", None)

In [4]:
class IMFCommodityClient:
    def __init__(self):
        self.client = sdmx.Client("IMF_DATA")

    def get_commodity(self, indicator, start=2021):
        return self.client.data("PCPS", key=f"G001.{indicator}.INDEX.M", params={'startPeriod': start})
    def get_indicators(self):
        return self.client.codelist()
    def get_interest(self, start=2021):
        return self.client.data("MFS_IR", key=f"*.*.*", params={'startPeriod': start}) 
        
class IMFTransformer:
    def __init__(self):
        pass
    def commodity_indicators(self, codelist):
        df = sdmx.to_pandas(codelist)["codelist"]["IMF.RES:CL_PCPS_INDICATOR"].reset_index()
        df = df.set_axis(["CommodityId", "CommodityName"], axis=1)
        return df
    def interest_indicators(self, codelist):
        df = sdmx.to_pandas(codelist)["codelist"]['IMF.STA:CL_MFS_IR_INDICATOR'].reset_index()
        df = df.set_axis(["InterestId", "InterestName"], axis=1)
        return df
    def check_result(self, result):
        if isinstance(result, pd.DataFrame) and not result.empty:
            return True
        else:
            return False
    def to_dataframe(self, result):
        return sdmx.to_pandas(result).reset_index()
    def parse_time_period(self, s: pd.Series) -> pd.Series:
        s = s.astype(str)
        # Quartale → Monat des Quartalsanfangs
        s = s.str.replace(r"(\d{4})-Q1", r"\1-01", regex=True)
        s = s.str.replace(r"(\d{4})-Q2", r"\1-04", regex=True)
        s = s.str.replace(r"(\d{4})-Q3", r"\1-07", regex=True)
        s = s.str.replace(r"(\d{4})-Q4", r"\1-10", regex=True)
        # Monate
        s = s.str.replace("-M", "-", regex=False)
        # reine Jahre
        s = s.where(~s.str.fullmatch(r"\d{4}"), s + "-01")
        # Datum erzeugen
        return pd.to_datetime(s + "-01")
    def commodity_timeries(self, timeseries, commodityname):
        df = sdmx.to_pandas(timeseries).reset_index()
        if isinstance(df, pd.DataFrame) and not df.empty:
            df["date"] = pd.to_datetime(df["TIME_PERIOD"].str.replace("-M", "-", regex=False), format="%Y-%m")
            df['commodity_name'] = commodityname
            df = df.rename(columns={"INDICATOR": "commodity_id"})
            df = df[['date', 'commodity_id', 'commodity_name', 'value']]
            return df
        return None

class IMFIngestor:
    def __init__(self):
        self.client = IMFCommodityClient()
        self.transformer = IMFTransformer()
    def run_commodity(self):
        list_of_commodities = []
        codelist = IMFCommodityClient().get_indicators()
        commodity_ids = IMFTransformer().commodity_indicators(codelist)
        for idx, row in commodity_ids.iterrows():
            id = row['CommodityId']
            commodity = row['CommodityName']
            ClientResult = IMFCommodityClient().get_commodity(id)
            timeseries = IMFTransformer().commodity_timeries(ClientResult, commodity)
            list_of_commodities.append(timeseries)
        return pd.concat(list_of_commodities)
    def run_interest(self):
        res = self.client.get_interest()
        df = self.transformer.to_dataframe(res)
        df['date'] = self.transformer.parse_time_period(df['TIME_PERIOD'])
        codelist = self.client.get_indicators()
        interest_lookup = self.transformer.interest_indicators(codelist)
        df = df.merge(interest_lookup[["InterestId", "InterestName"]], left_on="INDICATOR", right_on="InterestId", how="left")
        return df[['date', 'COUNTRY', 'INDICATOR', 'InterestName', 'FREQUENCY', 'value']]
                   


In [5]:
df = IMFIngestor().run_interest()

xml.Reader got no structure=… argument for StructureSpecificData


In [6]:
df.head()

,date,COUNTRY,INDICATOR,InterestName,FREQUENCY,value
0,2021-01-01,AFG,MFS135_FC_RT_PT_A_PT,"Deposit rate, Foreign Currency, Rate, Percent ...",M,1.263724
1,2021-02-01,AFG,MFS135_FC_RT_PT_A_PT,"Deposit rate, Foreign Currency, Rate, Percent ...",M,0.659519
2,2021-03-01,AFG,MFS135_FC_RT_PT_A_PT,"Deposit rate, Foreign Currency, Rate, Percent ...",M,1.702206
3,2021-04-01,AFG,MFS135_FC_RT_PT_A_PT,"Deposit rate, Foreign Currency, Rate, Percent ...",M,1.350953
4,2021-05-01,AFG,MFS135_FC_RT_PT_A_PT,"Deposit rate, Foreign Currency, Rate, Percent ...",M,1.368217
